# 2. Develop And Evaluate LLM Extraction Prompt

This notebook starts after notebook `1` has created the filtered-text human coding sample and the completed 500-row labelled development/test files. It runs the LLM extraction prompt on the labelled development sample, parses the batch output, evaluates the LLM results against the human labels, and writes comparison/evaluation outputs for prompt refinement.

## Run Prompt On Labelled Sample

Use this section to create a batch request from the labelled sample, submit it to the Batch API, download the completed output, and parse it into a `results.csv` file.


In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

import os
import re
import json
from datetime import datetime
from pathlib import Path

import pandas as pd

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

# Code is run from the OneDrive project code folder.
CODE_ROOT = REPO_ROOT
TEXT_MINING_CODE_ROOT = CODE_ROOT / "text_mining"

# Data is read from and written to the share-drive project folder.
SHARE_ROOT = DATA_ROOT
# DATA_ROOT is configured through project_config.
ROOT = DATA_ROOT

# =========================================================
# SETTINGS
# =========================================================
MODEL = "gpt-5.4-mini"
INPUT_FOLDER = DATA_ROOT / "training_data_database_identification" / "Output"
SPLIT_MANIFEST_FILE = DATA_ROOT / "training_data_database_identification" / "labelled_data" / "labelled500_development_test_split.csv"
TARGET_SPLIT = "development"  # set to "test" or None to ignore the split manifest

RUN_NAME = f"batch_output_trial_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
START_INDEX = 0
MAX_FILES = 5  # set to None to process all files from START_INDEX onward

BATCH_DIR = DATA_ROOT / "training_data_database_identification" / "batch_runs" / RUN_NAME
REQUESTS_FILE = BATCH_DIR / "requests.jsonl"
MANIFEST_FILE = BATCH_DIR / "manifest.csv"
BATCH_INFO_FILE = BATCH_DIR / "batch_info.json"
BATCH_OUTPUT_FILE = BATCH_DIR / "batch_output.jsonl"
BATCH_ERROR_FILE = BATCH_DIR / "batch_error.jsonl"
RESULTS_FILE = BATCH_DIR / "results.csv"

FILE_PATTERN = "*.txt"
MAX_INPUT_TOKENS = 200000
CHARS_PER_TOKEN_ESTIMATE = 4.0
COMPLETION_WINDOW = "24h"

# Optional Batch API token cost estimate placeholders for gpt-5.4-mini
PRICE_INPUT_PER_1M = 0.375
PRICE_OUTPUT_PER_1M = 2.25

# =========================================================
# OPENAI CLIENT
# =========================================================
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY")) if OpenAI is not None else None

# =========================================================
# PROMPT
# =========================================================
PROMPT_FILE = TEXT_MINING_CODE_ROOT / "prompts" / "data_source_extraction_system_prompt.txt"
SYSTEM_PROMPT = PROMPT_FILE.read_text(encoding="utf-8")

USER_PROMPT_TEMPLATE = """Methods text:

{text}
"""

TEXT_FORMAT = {
    "type": "json_schema",
    "name": "data_resource_extraction",
    "schema": {
        "type": "object",
        "properties": {
            "db_use": {
                "type": "integer",
                "enum": [0, 1]
            },
            "instances": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "descriptive_instance": {"type": "string"},
                        "data_resource": {"type": "string"},
                        "data_holder": {"type": "string"}
                    },
                    "required": ["descriptive_instance", "data_resource", "data_holder"],
                    "additionalProperties": False
                }
            },
            "reason_short": {
                "type": "string"
            }
        },
        "required": [
            "db_use",
            "instances",
            "reason_short"
        ],
        "additionalProperties": False
    },
    "strict": True
}

# =========================================================
# HELPERS
# =========================================================
def normalize_whitespace(text):
    text = str(text)
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def prepare_text(text, max_input_tokens=MAX_INPUT_TOKENS, chars_per_token=CHARS_PER_TOKEN_ESTIMATE):
    text = normalize_whitespace(text)
    max_chars = int(max_input_tokens * chars_per_token)
    if len(text) <= max_chars:
        return text, False
    return text[:max_chars], True


def normalize_label_value(value, strip_chars=";,."):
    if value is None:
        return ""

    value = re.sub(r"\s+", " ", str(value)).strip().strip(strip_chars)
    if not value:
        return ""

    if value.lower() == "manual":
        return "MANUAL"

    return value


def deduplicate_instances(instances):
    cleaned = []
    seen = set()

    for instance in instances or []:
        if not isinstance(instance, dict):
            continue

        descriptive_instance = normalize_label_value(instance.get("descriptive_instance", ""), strip_chars=",.")
        data_resource = normalize_label_value(instance.get("data_resource", "")) or "MANUAL"
        data_holder = normalize_label_value(instance.get("data_holder", "")) or "MANUAL"

        if not descriptive_instance and data_resource == "MANUAL" and data_holder == "MANUAL":
            continue

        normalized_instance = {
            "descriptive_instance": descriptive_instance,
            "data_resource": data_resource,
            "data_holder": data_holder,
        }
        key = (
            descriptive_instance.lower(),
            data_resource.lower(),
            data_holder.lower(),
        )
        if key not in seen:
            seen.add(key)
            cleaned.append(normalized_instance)

    return cleaned


def serialize_json_field(value):
    return json.dumps(value, ensure_ascii=False)


def estimate_tokens_from_chars(text, chars_per_token=CHARS_PER_TOKEN_ESTIMATE):
    return int(len(text) / chars_per_token)


def read_txt_file(filepath):
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            return f.read()
    except UnicodeDecodeError:
        with open(filepath, "r", encoding="latin-1") as f:
            return f.read()


def build_request_body(text):
    return {
        "model": MODEL,
        "input": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": USER_PROMPT_TEMPLATE.format(text=text)
            }
        ],
        "text": {"format": TEXT_FORMAT},
        "store": False,
    }


def extract_output_text(body):
    if body is None:
        return None

    output_text = body.get("output_text")
    if output_text:
        return output_text

    chunks = []
    for item in body.get("output", []):
        for content_item in item.get("content", []):
            if content_item.get("type") in {"output_text", "text"}:
                text_value = content_item.get("text")
                if isinstance(text_value, dict):
                    text_value = text_value.get("value")
                if text_value:
                    chunks.append(text_value)

    if chunks:
        return "".join(chunks)

    return None


def download_file_content(file_id, target_path):
    response = client.files.content(file_id)
    if hasattr(response, "read"):
        data = response.read()
    elif hasattr(response, "content"):
        data = response.content
    else:
        data = bytes(response)

    if isinstance(data, str):
        data = data.encode("utf-8")

    target_path.write_bytes(data)
    return target_path


In [ ]:
# =========================================================
# BUILD BATCH REQUEST FILE + MANIFEST
# =========================================================
all_files = sorted(INPUT_FOLDER.glob(FILE_PATTERN))
target_filenames = None
if TARGET_SPLIT is not None and str(TARGET_SPLIT).strip().lower() not in {"", "all", "none"}:
    split_df = pd.read_csv(SPLIT_MANIFEST_FILE, encoding="utf-8-sig")
    required_cols = {"filename", "split"}
    missing_cols = sorted(required_cols - set(split_df.columns))
    if missing_cols:
        raise ValueError(f"Split manifest is missing expected columns: {missing_cols}")

    target_filenames = set(
        split_df.loc[
            split_df["split"].astype(str).str.strip().str.lower() == str(TARGET_SPLIT).strip().lower(),
            "filename",
        ].dropna().astype(str)
    )
    if not target_filenames:
        raise ValueError(f"No filenames found for split={TARGET_SPLIT!r} in {SPLIT_MANIFEST_FILE}")
    all_files = [filepath for filepath in all_files if filepath.name in target_filenames]
    print(f"Filtered to split={TARGET_SPLIT!r}: {len(all_files)} txt files.")

selected_files = all_files[START_INDEX:]
if MAX_FILES is not None:
    selected_files = selected_files[:MAX_FILES]

if len(selected_files) == 0:
    raise ValueError(f"No txt files found in {INPUT_FOLDER} for the current slice and split={TARGET_SPLIT!r}.")

BATCH_DIR.mkdir(parents=True, exist_ok=True)

manifest_rows = []
with REQUESTS_FILE.open("w", encoding="utf-8") as fout:
    for row_index, filepath in enumerate(selected_files, start=1):
        raw_text = read_txt_file(filepath)
        normalized_text = normalize_whitespace(raw_text)
        cleaned_text, was_truncated = prepare_text(raw_text)
        input_tokens_est = estimate_tokens_from_chars(cleaned_text)
        custom_id = f"{START_INDEX + row_index:07d}__{filepath.stem}"

        manifest_rows.append({
            "custom_id": custom_id,
            "file_name": filepath.name,
            "file_path": str(filepath),
            "original_text_char_len": len(normalized_text),
            "text_char_len": len(cleaned_text),
            "was_truncated": was_truncated,
            "input_tokens_estimate": input_tokens_est,
        })

        request_record = {
            "custom_id": custom_id,
            "method": "POST",
            "url": "/v1/responses",
            "body": build_request_body(cleaned_text),
        }
        fout.write(json.dumps(request_record, ensure_ascii=False) + "\n")

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(MANIFEST_FILE, index=False)

estimated_total_input_tokens = int(manifest_df["input_tokens_estimate"].sum())
estimated_input_cost = (estimated_total_input_tokens / 1_000_000) * PRICE_INPUT_PER_1M

print(f"Selected files: {len(manifest_df)}")
print(f"Requests written to: {REQUESTS_FILE}")
print(f"Manifest written to: {MANIFEST_FILE}")
print(f"Estimated input tokens: {estimated_total_input_tokens}")
print(f"Estimated batch input-only cost: {estimated_input_cost:.4f}")

manifest_df.head()


In [ ]:
# =========================================================
# SUBMIT BATCH JOB
# =========================================================
if client is None:
    raise ImportError("openai is not installed in this notebook environment. Install the openai package before submitting a batch job.")

if not REQUESTS_FILE.exists():
    raise ValueError("Run the build cell first so the batch request file exists.")

with REQUESTS_FILE.open("rb") as f:
    uploaded_file = client.files.create(file=f, purpose="batch")

batch = client.batches.create(
    input_file_id=uploaded_file.id,
    endpoint="/v1/responses",
    completion_window=COMPLETION_WINDOW,
    metadata={
        "run_name": RUN_NAME,
        "model": MODEL,
        "start_index": str(START_INDEX),
        "max_files": "ALL" if MAX_FILES is None else str(MAX_FILES),
    },
)

batch_info = {
    "run_name": RUN_NAME,
    "model": MODEL,
    "input_file_id": uploaded_file.id,
    "batch_id": batch.id,
    "status": batch.status,
    "request_file": str(REQUESTS_FILE),
    "manifest_file": str(MANIFEST_FILE),
    "batch_output_file": str(BATCH_OUTPUT_FILE),
    "batch_error_file": str(BATCH_ERROR_FILE),
    "results_file": str(RESULTS_FILE),
    "start_index": START_INDEX,
    "max_files": MAX_FILES,
}

BATCH_INFO_FILE.write_text(json.dumps(batch_info, indent=2), encoding="utf-8")

print(f"Uploaded file id: {uploaded_file.id}")
print(f"Batch id: {batch.id}")
print(f"Batch status: {batch.status}")
print(f"Batch info saved to: {BATCH_INFO_FILE}")

batch_info


In [ ]:
# =========================================================
# CHECK STATUS + DOWNLOAD OUTPUT FILES WHEN AVAILABLE
# =========================================================
if client is None:
    raise ImportError("openai is not installed in this notebook environment. Install the openai package before checking batch status.")

if not BATCH_INFO_FILE.exists():
    raise ValueError("Run the submit cell first so the batch info file exists.")

batch_info = json.loads(BATCH_INFO_FILE.read_text(encoding="utf-8"))
batch = client.batches.retrieve(batch_info["batch_id"])

batch_info.update({
    "status": batch.status,
    "output_file_id": getattr(batch, "output_file_id", None),
    "error_file_id": getattr(batch, "error_file_id", None),
    "completed_at": getattr(batch, "completed_at", None),
    "failed_at": getattr(batch, "failed_at", None),
    "expired_at": getattr(batch, "expired_at", None),
    "request_counts": getattr(batch, "request_counts", None),
})

BATCH_INFO_FILE.write_text(json.dumps(batch_info, indent=2, default=str), encoding="utf-8")

print(f"Batch status: {batch.status}")

if getattr(batch, "output_file_id", None):
    download_file_content(batch.output_file_id, BATCH_OUTPUT_FILE)
    print(f"Downloaded batch output to: {BATCH_OUTPUT_FILE}")

if getattr(batch, "error_file_id", None):
    download_file_content(batch.error_file_id, BATCH_ERROR_FILE)
    print(f"Downloaded batch errors to: {BATCH_ERROR_FILE}")

batch_info


In [ ]:
# =========================================================
# PARSE BATCH OUTPUT INTO CSV
# =========================================================
if not MANIFEST_FILE.exists():
    raise ValueError("Manifest file not found. Run the build cell first.")

if not BATCH_OUTPUT_FILE.exists() and not BATCH_ERROR_FILE.exists():
    raise ValueError("No batch output or error file found yet. Run the status/download cell after the batch completes.")

manifest_records = pd.read_csv(MANIFEST_FILE).to_dict(orient="records")
manifest_by_id = {row["custom_id"]: row for row in manifest_records}


def make_base_row(base):
    return {
        "custom_id": base.get("custom_id"),
        "file_name": base.get("file_name"),
        "file_path": base.get("file_path"),
        "original_text_char_len": base.get("original_text_char_len"),
        "text_char_len": base.get("text_char_len"),
        "was_truncated": base.get("was_truncated"),
        "input_tokens_estimate": base.get("input_tokens_estimate"),
    }


rows = []
handled_ids = set()

if BATCH_OUTPUT_FILE.exists():
    with BATCH_OUTPUT_FILE.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            record = json.loads(line)
            custom_id = record.get("custom_id")
            handled_ids.add(custom_id)
            base = manifest_by_id.get(custom_id, {})
            response = record.get("response") or {}
            status_code = response.get("status_code")
            body = response.get("body") or {}
            output_text = extract_output_text(body)

            if status_code != 200 or not output_text:
                row = make_base_row(base)
                row.update({
                    "db_use": None,
                    "instances_json": None,
                    "descriptive_instances_json": None,
                    "data_resources_json": None,
                    "data_holders_json": None,
                    "n_instances": None,
                    "reason_short": None,
                    "input_tokens": None,
                    "output_tokens": None,
                    "total_tokens": None,
                    "response_status_code": status_code,
                    "status": f"response_error_{status_code}",
                    "error_message": "Missing output_text or non-200 response.",
                    "raw_output_text": output_text,
                })
                rows.append(row)
                continue

            try:
                parsed = json.loads(output_text)
            except Exception as e:
                row = make_base_row(base)
                row.update({
                    "db_use": None,
                    "instances_json": None,
                    "descriptive_instances_json": None,
                    "data_resources_json": None,
                    "data_holders_json": None,
                    "n_instances": None,
                    "reason_short": None,
                    "input_tokens": None,
                    "output_tokens": None,
                    "total_tokens": None,
                    "response_status_code": status_code,
                    "status": "parse_error",
                    "error_message": str(e),
                    "raw_output_text": output_text,
                })
                rows.append(row)
                continue

            instances = deduplicate_instances(parsed.get("instances", []))
            db_use = int(parsed.get("db_use", 0))

            if len(instances) > 0:
                db_use = 1

            descriptive_instances = [instance["descriptive_instance"] for instance in instances]
            data_resources = [instance["data_resource"] for instance in instances]
            data_holders = [instance["data_holder"] for instance in instances]

            usage = body.get("usage") or {}

            row = make_base_row(base)
            row.update({
                "db_use": db_use,
                "instances_json": serialize_json_field(instances),
                "descriptive_instances_json": serialize_json_field(descriptive_instances),
                "data_resources_json": serialize_json_field(data_resources),
                "data_holders_json": serialize_json_field(data_holders),
                "n_instances": len(instances),
                "reason_short": parsed.get("reason_short", ""),
                "input_tokens": usage.get("input_tokens"),
                "output_tokens": usage.get("output_tokens"),
                "total_tokens": usage.get("total_tokens"),
                "response_status_code": status_code,
                "status": "ok",
                "error_message": None,
                "raw_output_text": output_text,
            })
            rows.append(row)

if BATCH_ERROR_FILE.exists():
    with BATCH_ERROR_FILE.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            record = json.loads(line)
            custom_id = record.get("custom_id")
            handled_ids.add(custom_id)
            base = manifest_by_id.get(custom_id, {})
            error = record.get("error") or {}

            row = make_base_row(base)
            row.update({
                "db_use": None,
                "instances_json": None,
                "descriptive_instances_json": None,
                "data_resources_json": None,
                "data_holders_json": None,
                "n_instances": None,
                "reason_short": None,
                "input_tokens": None,
                "output_tokens": None,
                "total_tokens": None,
                "response_status_code": None,
                "status": f"batch_error: {error.get('code', 'unknown')}",
                "error_message": error.get("message", json.dumps(error)),
                "raw_output_text": None,
            })
            rows.append(row)

for custom_id, base in manifest_by_id.items():
    if custom_id in handled_ids:
        continue

    row = make_base_row(base)
    row.update({
        "db_use": None,
        "instances_json": None,
        "descriptive_instances_json": None,
        "data_resources_json": None,
        "data_holders_json": None,
        "n_instances": None,
        "reason_short": None,
        "input_tokens": None,
        "output_tokens": None,
        "total_tokens": None,
        "response_status_code": None,
        "status": "missing_from_batch_output",
        "error_message": "No success or error line found for this custom_id.",
        "raw_output_text": None,
    })
    rows.append(row)

results_df = pd.DataFrame(rows).sort_values("custom_id").reset_index(drop=True)
results_df.to_csv(RESULTS_FILE, index=False)

n_ok = (results_df["status"] == "ok").sum()
n_err = (results_df["status"] != "ok").sum()
n_pos = (results_df["db_use"] == 1).sum()

print(f"Saved parsed batch results to: {RESULTS_FILE}")
print(f"Rows processed successfully: {n_ok}")
print(f"Rows with errors or missing output: {n_err}")
print(f"Rows labelled positive: {n_pos}")

results_df.head()


## Evaluate Against Human Labels

Use this section after the batch output has been parsed. By default it evaluates the `results.csv` created by the run section above. To evaluate an older run, replace `LLM_RESULTS_FILE` with that run's `results.csv` path.


In [ ]:
import importlib.util

# =========================================================
# EVALUATION SETTINGS
# =========================================================
HUMAN_LABELS_FILE = ROOT / "training_data_database_identification" / "labelled_data" / "database_coding_from_html_1-500.csv"
SPLIT_MANIFEST_FILE = ROOT / "training_data_database_identification" / "labelled_data" / "labelled500_development_test_split.csv"
TARGET_SPLIT = "development"  # set to "test" or None to ignore the split manifest

# Default: evaluate the run produced by the batch/parsing section above.
# To evaluate a previous run, replace this with that run's results.csv path.
LLM_RESULTS_FILE = RESULTS_FILE

OUTPUT_DIR = ROOT / "training_data_database_identification" / "evaluation" / LLM_RESULTS_FILE.parent.name
OUTPUT_BASENAME = f"llm_vs_human_{TARGET_SPLIT or 'all'}"
SIMPLE_COMPARISON_FILE = OUTPUT_DIR / f"llm_human_simple_comparison_{TARGET_SPLIT or 'all'}.xlsx"

HUMAN_DONE_VALUE = "1"



def resolve_table_path(path):
    path = Path(path)
    if path.exists():
        return path

    for suffix in (".csv", ".xlsx", ".xls"):
        candidate = path.with_suffix(suffix)
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Could not find table file: {path}")


def read_table(path):
    resolved = resolve_table_path(path)
    if resolved.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(resolved)
    return pd.read_csv(resolved, encoding="utf-8-sig")

print(f"Human labels file: {HUMAN_LABELS_FILE}")
print(f"LLM results file: {LLM_RESULTS_FILE}")
print(f"Output directory: {OUTPUT_DIR}")


In [ ]:
from difflib import SequenceMatcher

# =========================================================
# HELPERS
# =========================================================
def parse_binary(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()
    if text in {"0", "0.0"}:
        return 0
    if text in {"1", "1.0"}:
        return 1
    return pd.NA


def split_sources(value):
    if pd.isna(value):
        return []

    text = str(value).strip()
    if not text:
        return []

    if text.startswith("["):
        try:
            parsed = json.loads(text)
            if isinstance(parsed, list):
                return [str(part).strip() for part in parsed if str(part).strip()]
        except Exception:
            pass

    return [part.strip() for part in text.split(";") if part.strip()]


def normalize_source_name(text):
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip(" ;,.")
    return text


def normalize_source_list(value):
    normalized = []
    seen = set()

    for item in split_sources(value):
        norm = normalize_source_name(item)
        if not norm:
            continue
        if norm in seen:
            continue
        seen.add(norm)
        normalized.append(norm)

    return normalized


def merge_unique_lists(*iterables):
    merged = []
    seen = set()

    for values in iterables:
        for item in values:
            norm = normalize_source_name(item)
            if not norm or norm in seen:
                continue
            seen.add(norm)
            merged.append(norm)

    return merged


def list_to_string(values):
    return "; ".join(values)


def intersection_string(left, right):
    return "; ".join(sorted(set(left) & set(right)))


def set_difference_string(left, right):
    return "; ".join(sorted(set(left) - set(right)))


def jaccard_similarity(left, right):
    left_set = set(left)
    right_set = set(right)

    if not left_set and not right_set:
        return 1.0

    union = left_set | right_set
    if not union:
        return 0.0

    return len(left_set & right_set) / len(union)


def safe_div(numerator, denominator):
    if denominator == 0:
        return pd.NA
    return numerator / denominator



def unique_source_names(values):
    """Return non-empty source names, deduplicated after strict normalization."""
    if isinstance(values, (list, tuple, set)):
        iterable = values
    else:
        if pd.isna(values):
            return []
        iterable = split_sources(values)

    output = []
    seen = set()
    for value in iterable:
        text = str(value).strip()
        norm = normalize_source_name(text)
        if not norm or norm in seen:
            continue
        seen.add(norm)
        output.append(text)
    return output


def source_list_to_json(values):
    return json.dumps(unique_source_names(values), ensure_ascii=False)


def normalize_source_for_fuzzy(text):
    text = normalize_source_name(text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\b(the|a|an)\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def acronym_for_source(text):
    words = re.findall(r"[a-zA-Z]+", str(text))
    stop_words = {"the", "and", "for", "from", "with", "data", "dataset", "database"}
    initials = [word[0].lower() for word in words if len(word) > 2 and word.lower() not in stop_words]
    if len(initials) < 2:
        return ""
    return "".join(initials)


def source_match_score(human_source, llm_source):
    human_strict = normalize_source_name(human_source)
    llm_strict = normalize_source_name(llm_source)
    if not human_strict or not llm_strict:
        return 0.0, ""

    if human_strict == llm_strict:
        return 1.0, "strict_exact"

    human_fuzzy = normalize_source_for_fuzzy(human_source)
    llm_fuzzy = normalize_source_for_fuzzy(llm_source)
    if not human_fuzzy or not llm_fuzzy:
        return 0.0, ""

    if human_fuzzy == llm_fuzzy:
        return 0.99, "normalized_exact"

    human_acronym = acronym_for_source(human_source)
    llm_acronym = acronym_for_source(llm_source)
    if human_acronym and human_acronym == llm_fuzzy:
        return 0.96, "human_acronym"
    if llm_acronym and llm_acronym == human_fuzzy:
        return 0.96, "llm_acronym"
    if human_acronym and llm_acronym and human_acronym == llm_acronym:
        return 0.95, "shared_acronym"

    shorter, longer = sorted([human_fuzzy, llm_fuzzy], key=len)
    if len(shorter) >= 8 and f" {shorter} " in f" {longer} ":
        return 0.93, "substring"

    similarity = SequenceMatcher(None, human_fuzzy, llm_fuzzy).ratio()
    if similarity >= 0.88:
        return similarity, "string_similarity"

    return 0.0, ""


def build_source_match_details(human_sources, llm_sources, fuzzy=False, drop_manual=False):
    human_sources = unique_source_names(human_sources)
    llm_sources = unique_source_names(llm_sources)

    if drop_manual:
        human_sources = [source for source in human_sources if normalize_source_name(source) != "manual"]
        llm_sources = [source for source in llm_sources if normalize_source_name(source) != "manual"]

    candidates = []
    for human_idx, human_source in enumerate(human_sources):
        for llm_idx, llm_source in enumerate(llm_sources):
            if fuzzy:
                score, method = source_match_score(human_source, llm_source)
            else:
                score = 1.0 if normalize_source_name(human_source) == normalize_source_name(llm_source) else 0.0
                method = "strict_exact" if score else ""

            if score > 0:
                candidates.append((score, method, human_idx, llm_idx, human_source, llm_source))

    candidates = sorted(candidates, key=lambda item: item[0], reverse=True)
    used_human = set()
    used_llm = set()
    matches = []

    for score, method, human_idx, llm_idx, human_source, llm_source in candidates:
        if human_idx in used_human or llm_idx in used_llm:
            continue
        used_human.add(human_idx)
        used_llm.add(llm_idx)
        matches.append({
            "human_source": human_source,
            "llm_source": llm_source,
            "match_method": method,
            "score": round(float(score), 4),
        })

    missing_human = [source for idx, source in enumerate(human_sources) if idx not in used_human]
    extra_llm = [source for idx, source in enumerate(llm_sources) if idx not in used_llm]

    return {
        "human_sources": human_sources,
        "llm_sources": llm_sources,
        "matches": matches,
        "missing_human": missing_human,
        "extra_llm": extra_llm,
        "match_count": len(matches),
        "full_human_capture": len(missing_human) == 0,
        "full_list_match": len(missing_human) == 0 and len(extra_llm) == 0,
    }


def details_to_json(details, key):
    return json.dumps(details[key], ensure_ascii=False)

def choose_excel_engine():
    for candidate in ("xlsxwriter", "openpyxl"):
        if importlib.util.find_spec(candidate) is not None:
            return candidate
    return None


In [ ]:
# =========================================================
# LOAD AND COMPARE
# =========================================================
human_df = read_table(HUMAN_LABELS_FILE)
human_df = human_df[human_df["done"].astype(str) == HUMAN_DONE_VALUE].copy()

split_df = read_table(SPLIT_MANIFEST_FILE)
split_keep_cols = ["doi", "filename", "split", "evaluation_eligible"]
missing_split_cols = sorted(set(split_keep_cols) - set(split_df.columns))
if missing_split_cols:
    raise ValueError(f"Split manifest is missing expected columns: {missing_split_cols}")

human_df = human_df.merge(split_df[split_keep_cols], on="doi", how="left", validate="one_to_one")
if TARGET_SPLIT is not None and str(TARGET_SPLIT).strip().lower() not in {"", "all", "none"}:
    human_df = human_df[
        human_df["split"].astype(str).str.strip().str.lower() == str(TARGET_SPLIT).strip().lower()
    ].copy()

human_df["human_db_use_label"] = human_df["db_use"].apply(parse_binary)
human_df["human_sources_list"] = human_df["databases"].apply(split_sources)
human_df["human_sources_normalized"] = human_df["databases"].apply(normalize_source_list)
human_df["human_n_sources"] = human_df["human_sources_list"].apply(len)

llm_df = read_table(LLM_RESULTS_FILE)
if "file_name" not in llm_df.columns and "filename" in llm_df.columns:
    llm_df["file_name"] = llm_df["filename"]
if "doi" not in llm_df.columns and "file_name" in llm_df.columns:
    doi_lookup = split_df[["doi", "filename"]].dropna().copy()
    doi_lookup["filename"] = doi_lookup["filename"].astype(str)
    llm_df = llm_df.merge(doi_lookup, left_on="file_name", right_on="filename", how="left")
    if "filename" in llm_df.columns:
        llm_df = llm_df.drop(columns=["filename"])
if "custom_id" not in llm_df.columns:
    llm_df["custom_id"] = llm_df.get("file_name", pd.Series(index=llm_df.index, dtype="object"))
if "db_use" in llm_df.columns and "has_external_data_source" not in llm_df.columns:
    llm_df["has_external_data_source"] = llm_df["db_use"]
if "data_resources_json" in llm_df.columns and "data_sources" not in llm_df.columns:
    llm_df["data_sources"] = llm_df["data_resources_json"]
if "data_holders_json" in llm_df.columns and "data_holders" not in llm_df.columns:
    llm_df["data_holders"] = llm_df["data_holders_json"]
if "n_instances" in llm_df.columns and "n_data_sources" not in llm_df.columns:
    llm_df["n_data_sources"] = llm_df["n_instances"]
if "response_status_code" not in llm_df.columns:
    llm_df["response_status_code"] = pd.NA
if "error_message" not in llm_df.columns:
    llm_df["error_message"] = llm_df["status"].where(~llm_df["status"].fillna("").eq("ok"), pd.NA)
llm_keep_cols = [
    "custom_id",
    "doi",
    "file_name",
    "file_path",
    "has_external_data_source",
    "data_sources",
    "data_holders",
    "n_data_sources",
    "reason_short",
    "input_tokens",
    "output_tokens",
    "total_tokens",
    "response_status_code",
    "status",
    "error_message",
]
missing_llm_cols = sorted(set(llm_keep_cols) - set(llm_df.columns))
if missing_llm_cols:
    raise ValueError(f"LLM results file is missing expected columns: {missing_llm_cols}")

llm_df = llm_df[llm_keep_cols].copy()
llm_df["llm_label"] = llm_df["has_external_data_source"].apply(parse_binary)
llm_df["llm_sources_list"] = llm_df["data_sources"].apply(split_sources)
llm_df["llm_sources_normalized"] = llm_df["data_sources"].apply(normalize_source_list)
llm_df["llm_holders_list"] = llm_df["data_holders"].apply(split_sources)
llm_df["llm_holders_normalized"] = llm_df["data_holders"].apply(normalize_source_list)
llm_df["llm_resource_or_holder_normalized"] = llm_df.apply(
    lambda row: merge_unique_lists(row["llm_sources_normalized"], row["llm_holders_normalized"]),
    axis=1,
)
llm_df["llm_n_sources_from_lists"] = llm_df["llm_sources_list"].apply(len)

comparison_df = human_df.merge(llm_df, on="doi", how="left", validate="one_to_one")
comparison_df["llm_ok"] = comparison_df["status"].fillna("").eq("ok")

comparison_df["binary_agreement"] = comparison_df.apply(
    lambda row: (row["human_db_use_label"] == row["llm_label"])
    if pd.notna(row["human_db_use_label"]) and pd.notna(row["llm_label"])
    else pd.NA,
    axis=1,
)

comparison_df["human_sources_normalized_str"] = comparison_df["human_sources_normalized"].apply(list_to_string)
comparison_df["llm_sources_normalized_str"] = comparison_df["llm_sources_normalized"].apply(list_to_string)
comparison_df["llm_holders_normalized_str"] = comparison_df["llm_holders_normalized"].apply(list_to_string)
comparison_df["llm_resource_or_holder_normalized_str"] = comparison_df["llm_resource_or_holder_normalized"].apply(list_to_string)
comparison_df["human_only_sources"] = comparison_df.apply(
    lambda row: set_difference_string(row["human_sources_normalized"], row["llm_sources_normalized"]),
    axis=1,
)
comparison_df["llm_only_sources"] = comparison_df.apply(
    lambda row: set_difference_string(row["llm_sources_normalized"], row["human_sources_normalized"]),
    axis=1,
)
comparison_df["human_codes_matched_via_resource_or_holder"] = comparison_df.apply(
    lambda row: intersection_string(row["human_sources_normalized"], row["llm_resource_or_holder_normalized"])
    if pd.notna(row["human_db_use_label"]) and row["human_db_use_label"] == 1
    else "",
    axis=1,
)
comparison_df["human_codes_missing_via_resource_or_holder"] = comparison_df.apply(
    lambda row: set_difference_string(row["human_sources_normalized"], row["llm_resource_or_holder_normalized"])
    if pd.notna(row["human_db_use_label"]) and row["human_db_use_label"] == 1
    else "",
    axis=1,
)
comparison_df["human_code_overlap_count_via_resource_or_holder"] = comparison_df.apply(
    lambda row: len(set(row["human_sources_normalized"]) & set(row["llm_resource_or_holder_normalized"]))
    if pd.notna(row["human_db_use_label"]) and row["human_db_use_label"] == 1
    else pd.NA,
    axis=1,
)
comparison_df["human_code_overlap_rate_via_resource_or_holder"] = comparison_df.apply(
    lambda row: safe_div(
        len(set(row["human_sources_normalized"]) & set(row["llm_resource_or_holder_normalized"])),
        len(row["human_sources_normalized"]),
    )
    if pd.notna(row["human_db_use_label"]) and row["human_db_use_label"] == 1
    else pd.NA,
    axis=1,
)
def is_positive_on_both_sides(row):
    return (
        pd.notna(row["human_db_use_label"])
        and pd.notna(row["llm_label"])
        and row["human_db_use_label"] == 1
        and row["llm_label"] == 1
    )


comparison_df["source_exact_match"] = comparison_df.apply(
    lambda row: (set(row["human_sources_normalized"]) == set(row["llm_sources_normalized"]))
    if is_positive_on_both_sides(row)
    else pd.NA,
    axis=1,
)
comparison_df["source_jaccard"] = comparison_df.apply(
    lambda row: jaccard_similarity(row["human_sources_normalized"], row["llm_sources_normalized"])
    if is_positive_on_both_sides(row)
    else pd.NA,
    axis=1,
)

def assign_comparison_case(row):
    if pd.isna(row["human_db_use_label"]):
        return "human_label_missing"

    if pd.isna(row["llm_label"]) or not row["llm_ok"]:
        return "llm_missing_or_error"

    if row["human_db_use_label"] == 1 and row["llm_label"] == 1:
        if row["source_exact_match"] is True:
            return "true_positive_source_match"
        return "true_positive_source_mismatch"

    if row["human_db_use_label"] == 0 and row["llm_label"] == 0:
        return "true_negative"

    if row["human_db_use_label"] == 0 and row["llm_label"] == 1:
        return "false_positive"

    if row["human_db_use_label"] == 1 and row["llm_label"] == 0:
        return "false_negative"

    return "other"


comparison_df["comparison_case"] = comparison_df.apply(assign_comparison_case, axis=1)
comparison_df["review_needed"] = comparison_df["comparison_case"].isin(
    {
        "human_label_missing",
        "llm_missing_or_error",
        "false_positive",
        "false_negative",
        "true_positive_source_mismatch",
    }
)

print(f"Human-coded rows loaded: {len(human_df)}")
print(f"LLM result rows loaded: {len(llm_df)}")
print(f"Comparison rows: {len(comparison_df)}")
comparison_df[[
    "doi",
    "human_db_use_label",
    "llm_label",
    "binary_agreement",
    "comparison_case",
    "human_databases" if "human_databases" in comparison_df.columns else "databases",
    "data_sources",
]].head()

## Source-Count Evaluation

This step compares the concrete data-source names extracted by the LLM with the source names in the human coding sheet. It reports both strict exact matches and a normalized/fuzzy comparison that allows minor punctuation, acronym, substring, and spelling variants. The row-level table is intended for manual inspection; the summary table is the one to cite.


In [ ]:
# =========================================================
# SOURCE-COUNT EVALUATION
# =========================================================
source_count_rows = []

for row_number, (_, row) in enumerate(comparison_df.iterrows(), start=1):
    human_sources = unique_source_names(row["human_sources_list"])
    llm_sources = unique_source_names(row["llm_sources_list"])

    exact_details = build_source_match_details(human_sources, llm_sources, fuzzy=False)
    fuzzy_details = build_source_match_details(human_sources, llm_sources, fuzzy=True)
    exact_no_manual_details = build_source_match_details(human_sources, llm_sources, fuzzy=False, drop_manual=True)
    fuzzy_no_manual_details = build_source_match_details(human_sources, llm_sources, fuzzy=True, drop_manual=True)

    source_count_rows.append({
        "row_number": row_number,
        "doi": row["doi"],
        "file_name": row.get("file_name", pd.NA),
        "split": row.get("split", pd.NA),
        "human_db_use": row["human_db_use_label"],
        "llm_db_use": row["llm_label"],
        "binary_outcome": row["comparison_case"],
        "human_sources_json": source_list_to_json(human_sources),
        "llm_sources_json": source_list_to_json(llm_sources),
        "human_source_count": len(human_sources),
        "llm_source_count": len(llm_sources),
        "exact_match_count": exact_details["match_count"],
        "exact_matches_json": details_to_json(exact_details, "matches"),
        "exact_missing_human_json": details_to_json(exact_details, "missing_human"),
        "exact_extra_llm_json": details_to_json(exact_details, "extra_llm"),
        "exact_full_list_match": exact_details["full_list_match"],
        "fuzzy_match_count": fuzzy_details["match_count"],
        "fuzzy_matches_json": details_to_json(fuzzy_details, "matches"),
        "fuzzy_missing_human_json": details_to_json(fuzzy_details, "missing_human"),
        "fuzzy_extra_llm_json": details_to_json(fuzzy_details, "extra_llm"),
        "fuzzy_full_human_capture": fuzzy_details["full_human_capture"],
        "fuzzy_full_list_match": fuzzy_details["full_list_match"],
        "human_source_count_no_manual": len(exact_no_manual_details["human_sources"]),
        "llm_source_count_no_manual": len(exact_no_manual_details["llm_sources"]),
        "exact_match_count_no_manual": exact_no_manual_details["match_count"],
        "fuzzy_match_count_no_manual": fuzzy_no_manual_details["match_count"],
        "human_sources_no_manual_json": details_to_json(exact_no_manual_details, "human_sources"),
        "llm_sources_no_manual_json": details_to_json(exact_no_manual_details, "llm_sources"),
        "human_comments": row.get("comments", pd.NA),
        "reason_short": row.get("reason_short", pd.NA),
    })

source_count_comparison_df = pd.DataFrame(source_count_rows)
source_count_eval_df = source_count_comparison_df[
    source_count_comparison_df["human_db_use"].isin([0, 1])
    & source_count_comparison_df["llm_db_use"].isin([0, 1])
].copy()
source_count_human_positive_df = source_count_eval_df[source_count_eval_df["human_db_use"] == 1].copy()
source_count_human_negative_df = source_count_eval_df[source_count_eval_df["human_db_use"] == 0].copy()


def summarize_source_name_matching(source_df, human_positive_df, human_negative_df, suffix="", label=""):
    human_count_col = f"human_source_count{suffix}"
    llm_count_col = f"llm_source_count{suffix}"
    exact_count_col = f"exact_match_count{suffix}"
    fuzzy_count_col = f"fuzzy_match_count{suffix}"

    human_positive_rows = len(human_positive_df)
    human_source_total = int(human_positive_df[human_count_col].fillna(0).sum())
    llm_source_total_positive = int(human_positive_df[llm_count_col].fillna(0).sum())
    llm_source_total_all = int(source_df[llm_count_col].fillna(0).sum())
    exact_matches = int(human_positive_df[exact_count_col].fillna(0).sum())
    fuzzy_matches = int(human_positive_df[fuzzy_count_col].fillna(0).sum())

    prefix = f"{label}_" if label else ""
    rows = [
        {"metric": f"{prefix}human_positive_rows", "value": human_positive_rows, "note": "Rows where the human coder marked external data-source use."},
        {"metric": f"{prefix}human_source_labels_total_on_human_positive_rows", "value": human_source_total, "note": "Total human-coded source labels on human-positive rows."},
        {"metric": f"{prefix}llm_source_labels_total_on_human_positive_rows", "value": llm_source_total_positive, "note": "Total LLM source labels on the same human-positive rows."},
        {"metric": f"{prefix}llm_source_labels_total_all_rows", "value": llm_source_total_all, "note": "Total LLM source labels on all evaluated rows."},
        {"metric": f"{prefix}strict_exact_source_matches", "value": exact_matches, "note": "One-to-one source-name matches after only case/space normalization."},
        {"metric": f"{prefix}strict_exact_source_recall", "value": safe_div(exact_matches, human_source_total), "note": "Matched human source labels / all human source labels."},
        {"metric": f"{prefix}strict_exact_source_precision_human_positive_rows", "value": safe_div(exact_matches, llm_source_total_positive), "note": "Matched LLM source labels / LLM source labels on human-positive rows."},
        {"metric": f"{prefix}strict_exact_source_precision_all_rows", "value": safe_div(exact_matches, llm_source_total_all), "note": "Matched LLM source labels / all LLM source labels, so emissions on human-negative rows count against precision."},
        {"metric": f"{prefix}normalized_fuzzy_source_matches", "value": fuzzy_matches, "note": "One-to-one source-name matches allowing normalized spelling, acronym, punctuation, and substring variants."},
        {"metric": f"{prefix}normalized_fuzzy_source_recall", "value": safe_div(fuzzy_matches, human_source_total), "note": "Matched human source labels / all human source labels."},
        {"metric": f"{prefix}normalized_fuzzy_source_precision_human_positive_rows", "value": safe_div(fuzzy_matches, llm_source_total_positive), "note": "Matched LLM source labels / LLM source labels on human-positive rows."},
        {"metric": f"{prefix}normalized_fuzzy_source_precision_all_rows", "value": safe_div(fuzzy_matches, llm_source_total_all), "note": "Matched LLM source labels / all LLM source labels, so emissions on human-negative rows count against precision."},
        {"metric": f"{prefix}human_negative_rows_with_llm_sources", "value": int((human_negative_df[llm_count_col].fillna(0) > 0).sum()), "note": "Human-negative rows where the model nevertheless emitted at least one source label."},
        {"metric": f"{prefix}llm_source_labels_on_human_negative_rows", "value": int(human_negative_df[llm_count_col].fillna(0).sum()), "note": "Total model source labels emitted on human-negative rows."},
        {"metric": f"{prefix}human_positive_rows_with_no_llm_sources", "value": int((human_positive_df[llm_count_col].fillna(0) == 0).sum()), "note": "Human-positive rows where the model emitted no source names."},
    ]

    if suffix == "":
        exact_full_list_count = int(human_positive_df["exact_full_list_match"].fillna(False).sum())
        fuzzy_full_human_capture_count = int(human_positive_df["fuzzy_full_human_capture"].fillna(False).sum())
        fuzzy_full_list_count = int(human_positive_df["fuzzy_full_list_match"].fillna(False).sum())
        rows.extend([
            {"metric": "strict_exact_full_list_matches_on_human_positive_rows", "value": f"{exact_full_list_count}/{human_positive_rows}", "note": "Whole source list exactly equal after case/space normalization."},
            {"metric": "strict_exact_full_list_match_rate", "value": safe_div(exact_full_list_count, human_positive_rows), "note": "Share of human-positive rows with identical source lists."},
            {"metric": "normalized_fuzzy_full_human_capture_rows", "value": f"{fuzzy_full_human_capture_count}/{human_positive_rows}", "note": "Rows where all human names were captured, allowing normalized/fuzzy variants."},
            {"metric": "normalized_fuzzy_full_human_capture_rate", "value": safe_div(fuzzy_full_human_capture_count, human_positive_rows), "note": "Share of human-positive rows where all human source labels were captured."},
            {"metric": "normalized_fuzzy_full_list_matches_on_human_positive_rows", "value": f"{fuzzy_full_list_count}/{human_positive_rows}", "note": "Rows where all human names were captured and no extra LLM names remained under fuzzy matching."},
            {"metric": "normalized_fuzzy_full_list_match_rate", "value": safe_div(fuzzy_full_list_count, human_positive_rows), "note": "Share of human-positive rows with fuzzy-equivalent source lists."},
        ])

    return rows

source_count_summary_rows = []
source_count_summary_rows.extend(
    summarize_source_name_matching(
        source_count_eval_df,
        source_count_human_positive_df,
        source_count_human_negative_df,
    )
)
source_count_summary_rows.extend(
    summarize_source_name_matching(
        source_count_eval_df,
        source_count_human_positive_df,
        source_count_human_negative_df,
        suffix="_no_manual",
        label="sensitivity_no_manual",
    )
)
source_count_summary_df = pd.DataFrame(source_count_summary_rows)

print(f"Source-count comparison rows: {len(source_count_comparison_df)}")
source_count_summary_df


In [ ]:
# =========================================================
# SUMMARY METRICS
# =========================================================
evaluation_df = comparison_df[
    comparison_df["human_db_use_label"].isin([0, 1])
    & comparison_df["llm_label"].isin([0, 1])
].copy()

tp = int(((evaluation_df["human_db_use_label"] == 1) & (evaluation_df["llm_label"] == 1)).sum())
tn = int(((evaluation_df["human_db_use_label"] == 0) & (evaluation_df["llm_label"] == 0)).sum())
fp = int(((evaluation_df["human_db_use_label"] == 0) & (evaluation_df["llm_label"] == 1)).sum())
fn = int(((evaluation_df["human_db_use_label"] == 1) & (evaluation_df["llm_label"] == 0)).sum())
n_eval = len(evaluation_df)

accuracy = safe_div(tp + tn, n_eval)
precision = safe_div(tp, tp + fp)
recall = safe_div(tp, tp + fn)
specificity = safe_div(tn, tn + fp)
f1 = safe_div(2 * tp, (2 * tp) + fp + fn)

human_positive_rate = safe_div(int((evaluation_df["human_db_use_label"] == 1).sum()), n_eval)
llm_positive_rate = safe_div(int((evaluation_df["llm_label"] == 1).sum()), n_eval)
expected_agreement = pd.NA
cohen_kappa = pd.NA
if n_eval > 0:
    expected_agreement = (
        (((tp + fp) / n_eval) * ((tp + fn) / n_eval))
        + (((fn + tn) / n_eval) * ((fp + tn) / n_eval))
    )
    if expected_agreement != 1:
        cohen_kappa = (accuracy - expected_agreement) / (1 - expected_agreement)

positive_both_df = evaluation_df[
    (evaluation_df["human_db_use_label"] == 1)
    & (evaluation_df["llm_label"] == 1)
].copy()
human_positive_df = evaluation_df[evaluation_df["human_db_use_label"] == 1].copy()
source_exact_count = int((positive_both_df["source_exact_match"] == True).sum())
source_exact_rate = safe_div(source_exact_count, len(positive_both_df))
source_jaccard_mean = positive_both_df["source_jaccard"].dropna().mean()
human_code_total_positive = int(human_positive_df["human_n_sources"].fillna(0).sum())
human_code_matched_positive = int(human_positive_df["human_code_overlap_count_via_resource_or_holder"].fillna(0).sum())
human_code_capture_micro_rate = safe_div(human_code_matched_positive, human_code_total_positive)
human_code_capture_mean_rate = human_positive_df["human_code_overlap_rate_via_resource_or_holder"].dropna().mean()
human_code_full_cover_count = int((human_positive_df["human_code_overlap_rate_via_resource_or_holder"] == 1).sum())
human_code_full_cover_rate = safe_div(human_code_full_cover_count, len(human_positive_df))
positive_both_human_code_total = int(positive_both_df["human_n_sources"].fillna(0).sum())
positive_both_human_code_matched = int(positive_both_df["human_code_overlap_count_via_resource_or_holder"].fillna(0).sum())
positive_both_human_code_capture_micro_rate = safe_div(positive_both_human_code_matched, positive_both_human_code_total)
positive_both_human_code_capture_mean_rate = positive_both_df["human_code_overlap_rate_via_resource_or_holder"].dropna().mean()

summary_rows = [
    {"metric": "human_done_rows", "value": len(human_df)},
    {"metric": "human_label_missing_rows", "value": int(comparison_df["human_db_use_label"].isna().sum())},
    {"metric": "llm_rows_loaded", "value": len(llm_df)},
    {"metric": "llm_ok_rows", "value": int(comparison_df["llm_ok"].sum())},
    {"metric": "rows_included_in_binary_evaluation", "value": n_eval},
    {"metric": "true_positive", "value": tp},
    {"metric": "true_negative", "value": tn},
    {"metric": "false_positive", "value": fp},
    {"metric": "false_negative", "value": fn},
    {"metric": "accuracy", "value": accuracy},
    {"metric": "precision", "value": precision},
    {"metric": "recall", "value": recall},
    {"metric": "specificity", "value": specificity},
    {"metric": "f1", "value": f1},
    {"metric": "human_positive_rate", "value": human_positive_rate},
    {"metric": "llm_positive_rate", "value": llm_positive_rate},
    {"metric": "cohen_kappa", "value": cohen_kappa},
    {"metric": "positive_on_both_sides", "value": len(positive_both_df)},
    {"metric": "source_exact_match_count", "value": source_exact_count},
    {"metric": "source_exact_match_rate_given_positive_both", "value": source_exact_rate},
    {"metric": "mean_source_jaccard_given_positive_both", "value": source_jaccard_mean},
    {"metric": "human_labeled_codes_total_positive_rows", "value": human_code_total_positive},
    {"metric": "human_labeled_codes_matched_via_resource_or_holder_total", "value": human_code_matched_positive},
    {"metric": "human_labeled_code_capture_micro_rate_via_resource_or_holder", "value": human_code_capture_micro_rate},
    {"metric": "mean_human_labeled_code_capture_rate_via_resource_or_holder", "value": human_code_capture_mean_rate},
    {"metric": "human_positive_rows_with_full_code_capture_via_resource_or_holder", "value": human_code_full_cover_count},
    {"metric": "human_positive_rows_full_code_capture_rate_via_resource_or_holder", "value": human_code_full_cover_rate},
    {"metric": "positive_both_human_labeled_code_capture_micro_rate_via_resource_or_holder", "value": positive_both_human_code_capture_micro_rate},
    {"metric": "positive_both_mean_human_labeled_code_capture_rate_via_resource_or_holder", "value": positive_both_human_code_capture_mean_rate},
]
source_count_summary_metric_lookup = source_count_summary_df.set_index("metric")["value"].to_dict()
summary_rows.extend([
    {"metric": "source_count_human_source_labels_total_on_human_positive_rows", "value": source_count_summary_metric_lookup.get("human_source_labels_total_on_human_positive_rows")},
    {"metric": "source_count_llm_source_labels_total_on_human_positive_rows", "value": source_count_summary_metric_lookup.get("llm_source_labels_total_on_human_positive_rows")},
    {"metric": "source_count_llm_source_labels_total_all_rows", "value": source_count_summary_metric_lookup.get("llm_source_labels_total_all_rows")},
    {"metric": "source_count_strict_exact_source_recall", "value": source_count_summary_metric_lookup.get("strict_exact_source_recall")},
    {"metric": "source_count_normalized_fuzzy_source_recall", "value": source_count_summary_metric_lookup.get("normalized_fuzzy_source_recall")},
    {"metric": "source_count_normalized_fuzzy_source_precision_human_positive_rows", "value": source_count_summary_metric_lookup.get("normalized_fuzzy_source_precision_human_positive_rows")},
])
summary_df = pd.DataFrame(summary_rows)

case_counts_df = (
    comparison_df["comparison_case"]
    .value_counts(dropna=False)
    .rename_axis("comparison_case")
    .reset_index(name="n_rows")
)

summary_df


In [ ]:
# =========================================================
# SAVE OUTPUTS
# =========================================================
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_csv = OUTPUT_DIR / f"{OUTPUT_BASENAME}_summary.csv"
case_counts_csv = OUTPUT_DIR / f"{OUTPUT_BASENAME}_case_counts.csv"
row_comparison_csv = OUTPUT_DIR / f"{OUTPUT_BASENAME}_row_comparison.csv"
review_cases_csv = OUTPUT_DIR / f"{OUTPUT_BASENAME}_review_cases.csv"
binary_disagreements_csv = OUTPUT_DIR / f"{OUTPUT_BASENAME}_binary_disagreements.csv"
source_count_summary_csv = OUTPUT_DIR / f"{OUTPUT_BASENAME}_source_count_summary.csv"
source_count_comparison_csv = OUTPUT_DIR / f"{OUTPUT_BASENAME}_source_count_comparison.csv"
workbook_path = OUTPUT_DIR / f"{OUTPUT_BASENAME}.xlsx"

comparison_export = comparison_df.copy()
comparison_export = comparison_export.rename(columns={
    "databases": "human_databases",
    "comments": "human_comments",
    "done": "human_done",
})

comparison_export["human_sources_list_str"] = comparison_export["human_sources_list"].apply(list_to_string)
comparison_export["llm_sources_list_str"] = comparison_export["llm_sources_list"].apply(list_to_string)
comparison_export["llm_holders_list_str"] = comparison_export["llm_holders_list"].apply(list_to_string)
comparison_export["llm_resource_or_holder_list_str"] = comparison_export["llm_resource_or_holder_normalized"].apply(list_to_string)

drop_list_cols = [
    "human_sources_list",
    "human_sources_normalized",
    "llm_sources_list",
    "llm_sources_normalized",
    "llm_holders_list",
    "llm_holders_normalized",
    "llm_resource_or_holder_normalized",
]
comparison_export = comparison_export.drop(columns=drop_list_cols)

preferred_column_order = [
    "doi",
    "custom_id",
    "file_name",
    "file_path",
    "human_done",
    "human_db_use_label",
    "llm_label",
    "binary_agreement",
    "comparison_case",
    "review_needed",
    "human_databases",
    "data_sources",
    "data_holders",
    "human_sources_list_str",
    "llm_sources_list_str",
    "llm_holders_list_str",
    "llm_resource_or_holder_list_str",
    "human_sources_normalized_str",
    "llm_sources_normalized_str",
    "llm_holders_normalized_str",
    "llm_resource_or_holder_normalized_str",
    "human_only_sources",
    "llm_only_sources",
    "human_codes_matched_via_resource_or_holder",
    "human_codes_missing_via_resource_or_holder",
    "human_code_overlap_count_via_resource_or_holder",
    "human_code_overlap_rate_via_resource_or_holder",
    "source_exact_match",
    "source_jaccard",
    "human_n_sources",
    "n_data_sources",
    "reason_short",
    "status",
    "error_message",
    "response_status_code",
    "input_tokens",
    "output_tokens",
    "total_tokens",
    "human_comments",
]
remaining_cols = [col for col in comparison_export.columns if col not in preferred_column_order]
comparison_export = comparison_export[preferred_column_order + remaining_cols]

review_cases_df = comparison_export[comparison_export["review_needed"]].copy()
binary_disagreements_df = comparison_export[
    comparison_export["comparison_case"].isin(["false_positive", "false_negative"])
].copy()

summary_df.to_csv(summary_csv, index=False)
case_counts_df.to_csv(case_counts_csv, index=False)
comparison_export.to_csv(row_comparison_csv, index=False)
review_cases_df.to_csv(review_cases_csv, index=False)
binary_disagreements_df.to_csv(binary_disagreements_csv, index=False)
source_count_summary_df.to_csv(source_count_summary_csv, index=False)
source_count_comparison_df.to_csv(source_count_comparison_csv, index=False)

excel_engine = choose_excel_engine()
if excel_engine is not None:
    with pd.ExcelWriter(workbook_path, engine=excel_engine) as writer:
        summary_df.to_excel(writer, index=False, sheet_name="summary")
        case_counts_df.to_excel(writer, index=False, sheet_name="case_counts")
        comparison_export.to_excel(writer, index=False, sheet_name="row_comparison")
        review_cases_df.to_excel(writer, index=False, sheet_name="review_cases")
        binary_disagreements_df.to_excel(writer, index=False, sheet_name="binary_disagreements")
        source_count_summary_df.to_excel(writer, index=False, sheet_name="source_count_summary")
        source_count_comparison_df.to_excel(writer, index=False, sheet_name="source_count_comparison")
    print(f"Workbook saved to: {workbook_path}")
else:
    print("No Excel engine found in this environment. CSV outputs were still saved.")

print(f"Summary CSV: {summary_csv}")
print(f"Case counts CSV: {case_counts_csv}")
print(f"Row comparison CSV: {row_comparison_csv}")
print(f"Review cases CSV: {review_cases_csv}")
print(f"Binary disagreements CSV: {binary_disagreements_csv}")
print(f"Source-count summary CSV: {source_count_summary_csv}")
print(f"Source-count comparison CSV: {source_count_comparison_csv}")

review_cases_df.head()


## Compact Side-By-Side Comparison Sheet

This creates a smaller review workbook from the detailed comparison table generated above.


In [ ]:
# =========================================================
# SAVE COMPACT REVIEW SHEET
# =========================================================
SIMPLE_COMPARISON_FILE.parent.mkdir(parents=True, exist_ok=True)
excel_engine = choose_excel_engine()

if excel_engine is None:
    raise ImportError("No Excel engine found. Install xlsxwriter or openpyxl in the notebook environment.")

simple_columns = [
    "doi",
    "filename",
    "file_name",
    "split",
    "evaluation_eligible",
    "human_db_use_label",
    "llm_label",
    "binary_agreement",
    "comparison_case",
    "review_needed",
    "databases",
    "comments",
    "data_sources",
    "llm_only_sources",
    "human_only_sources",
    "source_jaccard",
    "n_data_sources",
    "human_n_sources",
    "reason_short",
]
simple_columns = [column for column in simple_columns if column in comparison_df.columns]

simple_comparison_df = comparison_df[simple_columns].copy()
simple_comparison_df = simple_comparison_df.rename(columns={
    "databases": "human_databases",
    "comments": "human_comments",
    "data_sources": "llm_data_sources",
    "reason_short": "llm_reason_short",
})

source_count_compact_columns = [
    "doi",
    "human_source_count",
    "llm_source_count",
    "exact_match_count",
    "fuzzy_match_count",
    "fuzzy_full_human_capture",
    "fuzzy_full_list_match",
    "exact_missing_human_json",
    "exact_extra_llm_json",
    "fuzzy_missing_human_json",
    "fuzzy_extra_llm_json",
]
source_count_compact_df = source_count_comparison_df[source_count_compact_columns].copy()

with pd.ExcelWriter(SIMPLE_COMPARISON_FILE, engine=excel_engine) as writer:
    simple_comparison_df.to_excel(writer, index=False, sheet_name="comparison")
    source_count_summary_df.to_excel(writer, index=False, sheet_name="source_count_summary")
    source_count_compact_df.to_excel(writer, index=False, sheet_name="source_count_rows")

print(f"Saved simple comparison sheet to: {SIMPLE_COMPARISON_FILE}")
simple_comparison_df.head()
